# FashionMNIST multitask learning (class + price)

This notebook shows a simple multitask setup using `torchvision.datasets.FashionMNIST`:
- Task 1: classify the clothing item (`10` classes)
- Task 2: predict a synthetic price (regression)

The price is generated from class-dependent base prices plus uniform noise.

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

BASE_PRICE_BY_CLASS = {
    0: 18.0,
    1: 28.0,
    2: 35.0,
    3: 42.0,
    4: 55.0,
    5: 22.0,
    6: 25.0,
    7: 65.0,
    8: 48.0,
    9: 72.0,
}

def assign_price(label: int, base_price_by_class: dict[int, float], noise_low: float = -4.0, noise_high: float = 4.0, generator: torch.Generator | None = None) -> float:
    """Assign a synthetic price from a class->price map with uniform noise."""
    base_price = base_price_by_class[int(label)]
    noise = torch.empty(1).uniform_(noise_low, noise_high, generator=generator).item()
    return max(1.0, base_price + noise)

for i, name in enumerate(CLASS_NAMES):
    print(f'{i:>2} {name:<12} base=${BASE_PRICE_BY_CLASS[i]:.2f}')

In [8]:
class FashionMNISTWithPrice(Dataset):
    def __init__(
        self,
        root: str,
        train: bool,
        transform=None,
        download: bool = True,
        base_price_by_class: dict[int, float] | None = None,
        noise_low: float = -4.0,
        noise_high: float = 4.0,
        seed: int = 123,
    ):
        self.ds = datasets.FashionMNIST(root=root, train=train, transform=transform, download=download)
        self.base_price_by_class = base_price_by_class or BASE_PRICE_BY_CLASS

        g = torch.Generator().manual_seed(seed + (0 if train else 10_000))
        labels = self.ds.targets.tolist()
        self.prices = torch.tensor([
            assign_price(y, self.base_price_by_class, noise_low, noise_high, generator=g)
            for y in labels
        ], dtype=torch.float32)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx: int):
        image, label = self.ds[idx]
        price = self.prices[idx]
        return image, label, price

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

full_train = FashionMNISTWithPrice(root='./data', train=True, transform=transform, download=True)
test_ds = FashionMNISTWithPrice(root='./data', train=False, transform=transform, download=True)

train_ds, val_ds = random_split(full_train, [50_000, 10_000], generator=torch.Generator().manual_seed(42))

batch_size = 128
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=torch.cuda.is_available())

print('train batches:', len(train_loader), 'val batches:', len(val_loader), 'test batches:', len(test_loader))

train batches: 391 val batches: 79 test batches: 79


In [9]:
class MultiTaskConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.backbone = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.class_head = nn.Linear(128, 10)
        self.price_head = nn.Linear(128, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.backbone(x)
        class_logits = self.class_head(x)
        price_pred = self.price_head(x).squeeze(1)
        return class_logits, price_pred

model = MultiTaskConvNet().to(device)
ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

MultiTaskConvNet(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (backbone): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
  )
  (class_head): Linear(in_features=128, out_features=10, bias=True)
  (price_head): Linear(in_features=128, out_features=1, bias=True)
)


In [10]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    n = 0
    correct = 0
    price_abs_err = 0.0

    for x, y, price in loader:
        x = x.to(device)
        y = y.to(device)
        price = price.to(device)

        class_logits, price_pred = model(x)
        pred = class_logits.argmax(dim=1)

        correct += (pred == y).sum().item()
        price_abs_err += torch.abs(price_pred - price).sum().item()
        n += x.size(0)

    return {
        'acc': correct / n,
        'mae': price_abs_err / n,
    }

def train(model, train_loader, val_loader, epochs=5, regression_weight=0.02):
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for x, y, price in train_loader:
            x = x.to(device)
            y = y.to(device)
            price = price.to(device)

            class_logits, price_pred = model(x)
            loss_cls = ce_loss(class_logits, y)
            loss_reg = mse_loss(price_pred, price)
            loss = loss_cls + regression_weight * loss_reg

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate(model, val_loader)
        print(
            f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | '
            f'val_acc={val_metrics["acc"]:.4f} | val_mae=${val_metrics["mae"]:.2f}'
        )

In [11]:
train(model, train_loader, val_loader, epochs=5, regression_weight=0.02)

Epoch 01 | train_loss=4.6262 | val_acc=0.7959 | val_mae=$7.52
Epoch 02 | train_loss=2.3343 | val_acc=0.8445 | val_mae=$6.06
Epoch 03 | train_loss=2.0144 | val_acc=0.8557 | val_mae=$5.54
Epoch 04 | train_loss=1.8457 | val_acc=0.8640 | val_mae=$5.25
Epoch 05 | train_loss=1.7114 | val_acc=0.8705 | val_mae=$5.04


In [12]:
test_metrics = evaluate(model, test_loader)
print(f"Test accuracy: {test_metrics['acc']:.4f}")
print(f"Test price MAE: ${test_metrics['mae']:.2f}")

Test accuracy: 0.8659
Test price MAE: $5.19


In [13]:
# Quick sample predictions
model.eval()
x, y, p = next(iter(test_loader))
x, y, p = x.to(device), y.to(device), p.to(device)

with torch.no_grad():
    class_logits, price_pred = model(x)
    y_hat = class_logits.argmax(dim=1)

for i in range(10):
    true_label = CLASS_NAMES[y[i].item()]
    pred_label = CLASS_NAMES[y_hat[i].item()]
    print(
        f'{i:02d} true={true_label:<12} pred={pred_label:<12} '
        f'true_price=${p[i].item():>5.2f} pred_price=${price_pred[i].item():>5.2f}'
    )

00 true=Ankle boot   pred=Ankle boot   true_price=$69.00 pred_price=$67.82
01 true=Pullover     pred=Pullover     true_price=$36.05 pred_price=$33.72
02 true=Trouser      pred=Trouser      true_price=$24.92 pred_price=$28.18
03 true=Trouser      pred=Trouser      true_price=$24.02 pred_price=$29.35
04 true=Shirt        pred=Shirt        true_price=$22.38 pred_price=$24.76
05 true=Trouser      pred=Trouser      true_price=$31.50 pred_price=$28.48
06 true=Coat         pred=Coat         true_price=$55.15 pred_price=$32.15
07 true=Shirt        pred=Shirt        true_price=$24.13 pred_price=$28.28
08 true=Sandal       pred=Sandal       true_price=$19.54 pred_price=$31.39
09 true=Sneaker      pred=Sneaker      true_price=$63.22 pred_price=$74.06
